In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
recordings_path = "#path"  
files = sorted(os.listdir(recordings_path))
print(f"Found {len(files)} files:")
for f in files:
    print(f)

Found 23 files:
01_koramangala_quiet.wav
02_indiranagar_noisy.wav
03_whitefield_quiet.wav
04_electroniccity_noisy.wav
05_marathahalli_whispered.wav
06_jayanagar_quiet.wav
07_rajajinagar_noisy.wav
08_hebbal_rushed.wav
09_yelahanka_quiet.wav
10_banashankari_noisy.wav
11_hsrlayout_quiet.wav
12_btmlayout_noisy.wav
13_majestic_rushed.wav
14_silkboard_noisy.wav
15_bellandur_quiet.wav
16_sarjapur_whispered.wav
17_bommanahalli_noisy.wav
18_krpuram_rushed.wav
19_peenya_quiet.wav
20_yeshwanthpur_noisy.wav
21_koramangala_english_quiet.wav
22_indiranagar_english_noisy.wav
metadata.csv


In [ ]:
!pip install faster-whisper transformers accelerate -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 100.1 MB/s eta 0:00:00


In [ ]:
!pip install transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1" -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 105.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.6 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login
login(token="#token")  # get from huggingface.co/settings/tokens

In [ ]:
from transformers import AutoModel
import torch, torchaudio

model = AutoModel.from_pretrained(
    "ai4bharat/indic-conformer-600m-multilingual",
    trust_remote_code=True
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("Model loaded on:", device)

Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Model loaded on: cuda


In [ ]:
import os, time, pandas as pd

recordings_path = "/content/drive/MyDrive/Vaahan assesment/recordings"
results = []

for filename in sorted(os.listdir(recordings_path)):
    if not filename.endswith(".wav"):
        continue
    filepath = os.path.join(recordings_path, filename)
    try:
        wav, sr = torchaudio.load(filepath)
        wav = torch.mean(wav, dim=0, keepdim=True)
        if sr != 16000:
            wav = torchaudio.transforms.Resample(sr, 16000)(wav)
        wav = wav.to(device)

        start = time.time()
        transcript = model(wav, "hi", "ctc")
        latency = round(time.time() - start, 3)

        print(f"{filename}: {transcript}")
        results.append({
            "filename": filename,
            "model": "indicconformer-600m",
            "hypothesis": transcript,
            "latency_sec": latency
        })
    except Exception as e:
        print(f"ERROR on {filename}: {e}")
        results.append({
            "filename": filename,
            "model": "indicconformer-600m",
            "hypothesis": "",
            "latency_sec": None
        })

df = pd.DataFrame(results)
print(df)

01_koramangala_quiet.wav: हन बाई मैन कोरमंगला में रहता हूं नयर सोनी वेड सिग्नल
02_indiranagar_noisy.wav: मेरे घर इंदिराानगर में हैं हंड्रेड फीट रोड के पास
03_whitefield_quiet.wav: मैं व्हाइट फील्ड से हूं ऑफिस भी वही है
04_electroniccity_noisy.wav: हा सार इलेक्ट्रॉणिक सिटी सेेस्टम ने काम करर्ता हुई
05_marathahalli_whispered.wav: अभी मारतह्ली ब्रिज के पासन थोड़ा बिजी एरिया है
06_jayanagar_quiet.wav: मैम जयनगर मेम रहता हून फोर् प्लॉक
07_rajajinagar_noisy.wav: राजजिनगर में एक रोम ढूंढ रहा है बजट काम है
08_hebbal_rushed.wav: हा हा है् बाल फ्लाईओवर के पास रहता हूं जल्दी बताओ
09_yelahanka_quiet.wav: येलाहंगा में एक नया फ्लैट लिया है हमने
10_banashankari_noisy.wav: बनशगिरी सेकेंड स्टेज में रहता हुबई
11_hsrlayout_quiet.wav: जी हँ मेरा एड्रेस एचएसआर लेआउट है सेक्टर टू
12_btmlayout_noisy.wav: बीटीएम लेवट मेंैं हूं यार स्ेज वन साइड
13_majestic_rushed.wav: मैं अभी माजेस्टिक बस के पासू थोड़ा वेट करूँ
14_silkboard_noisy.wav: यार सिल्कबोर्ड प बहुत जाान है एक घांटे से खदा हो
15_bellandur_quiet.wav: बेल

In [ ]:
df.to_csv("/content/drive/MyDrive/Vaahan assesment/indicconformer_results.csv", index=False)
print("Saved")

Saved
